# Semana 6: Práctica. La diversificación medida con datos

**Curso:** Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/06_riesgo_rendimiento/clase06_practica.ipynb)

Hoy llevamos las fórmulas de la teoría a un universo real de 14 acciones (locales de la BVL, peruanas que cotizan en Nueva York y grandes de Estados Unidos) para responder con datos tres preguntas: quién es quién en el mapa riesgo-retorno, qué tan juntas se mueven, y cuánto riesgo desaparece de verdad al diversificar.

**Requisito previo:** `git pull` en tu fork para tener `utils/finanzas.py` actualizado (esta semana se agregan las funciones de portafolio).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from utils.finanzas import (retornos, portafolio_dos_activos, peso_minima_varianza,
                            riesgo_portafolio, riesgo_equiponderado)

GRUPOS = {
    "BVL (soles)":    ["ALICORC1.LM", "FERREYC1.LM", "CPACASC1.LM", "LUSURC1.LM"],
    "Peru en NYSE":   ["BAP", "BVN", "SCCO", "IFS"],
    "Estados Unidos": ["AAPL", "MSFT", "JNJ", "KO", "XOM", "JPM"],
}
REFERENCIAS = ["SPY", "EPU"]      # ETF del S&P 500 y del MSCI Peru
COLORES = {"BVL (soles)": "#E36C0A", "Peru en NYSE": "#000080", "Estados Unidos": "#006E6E"}

UNIVERSO = [t for g in GRUPOS.values() for t in g]

## 1. Descargar precios y llevar todo a una sola moneda

Bajamos 5 años de precios mensuales ajustados (por dividendos y splits, así el retorno es total). Dos cuidados de un universo mixto:

- **Moneda.** Las acciones de la BVL cotizan en soles y el resto en dólares. Un portafolio se mide en una sola moneda, así que convertimos los precios locales a dólares con el tipo de cambio de cada mes (`PEN=X` es soles por dólar). El retorno en dólares de una acción local mezcla el retorno de la acción y el del sol.
- **Mes incompleto.** La última fila es el mes en curso, que todavía no cierra: la descartamos.

In [ ]:
bruto = yf.download(UNIVERSO + REFERENCIAS + ["PEN=X"], period="5y", interval="1mo",
                    auto_adjust=True, progress=False)["Close"]
bruto = bruto.iloc[:-1]                                   # fuera el mes en curso

tc = bruto["PEN=X"].ffill()                               # soles por dolar
precios = bruto.drop(columns="PEN=X")
locales = [t for t in precios.columns if t.endswith(".LM")]
precios[locales] = precios[locales].div(tc, axis=0)       # de soles a dolares

# Defensa: si a un ticker le faltan muchos datos, se reporta y se excluye
faltantes = [t for t in precios.columns if precios[t].notna().sum() < 48]
if faltantes:
    print("Excluidos por datos insuficientes:", faltantes)
precios = precios.drop(columns=faltantes).ffill()
GRUPOS = {g: [t for t in ts if t in precios.columns] for g, ts in GRUPOS.items()}
UNIVERSO = [t for g in GRUPOS.values() for t in g]

ret_todo = retornos(precios)                              # retornos mensuales simples, en USD
ret = ret_todo[UNIVERSO]
print(f"{ret.shape[1]} acciones, {ret.shape[0]} meses: de {ret.index[0].date()} a {ret.index[-1].date()}")

## 2. El mapa riesgo-retorno

Anualizamos como en la teoría: la media por 12 y la volatilidad por $\sqrt{12}$. Agregamos la media geométrica mensual, también multiplicada por 12 para que sea comparable con la aritmética, y así ver cuánto se come la volatilidad.

In [ ]:
tabla = pd.DataFrame({
    "retorno medio": ret_todo.mean() * 12,
    "volatilidad": ret_todo.std() * np.sqrt(12),
    "retorno geometrico": ((1 + ret_todo).prod() ** (1 / len(ret_todo)) - 1) * 12,
})
tabla["brecha arit - geom"] = tabla["retorno medio"] - tabla["retorno geometrico"]
(tabla * 100).round(1).sort_values("volatilidad")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for grupo, tickers in GRUPOS.items():
    ax.scatter(tabla.loc[tickers, "volatilidad"] * 100, tabla.loc[tickers, "retorno medio"] * 100,
               s=60, color=COLORES[grupo], label=grupo)
ax.scatter(tabla.loc[REFERENCIAS, "volatilidad"] * 100, tabla.loc[REFERENCIAS, "retorno medio"] * 100,
           s=90, marker="D", color="black", label="ETF de referencia")
for t in tabla.index:
    ax.annotate(t.replace(".LM", ""), (tabla.loc[t, "volatilidad"] * 100, tabla.loc[t, "retorno medio"] * 100),
                xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("Volatilidad anualizada (%)"); ax.set_ylabel("Retorno medio anualizado (%)")
ax.set_title("Mapa riesgo-retorno del universo (60 meses, retornos mensuales en USD)")
ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

**Preguntas de discusión:** ¿más riesgo trajo más retorno en esta muestra? (No tiene por qué: 5 años es una sola realización de la historia y la media muestral es muy ruidosa.) ¿Dónde caen los ETF respecto de las acciones individuales? Un ETF ya es un portafolio: mira su volatilidad contra la de sus componentes. ¿Qué acción tiene la mayor brecha entre media aritmética y geométrica, y por qué coincide con la más volátil?

## 3. La matriz de correlaciones

Aquí está la materia prima de la diversificación.

In [ ]:
corr = ret.corr()

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
etiquetas = [t.replace(".LM", "") for t in corr.columns]
ax.set_xticks(range(len(etiquetas))); ax.set_xticklabels(etiquetas, rotation=90)
ax.set_yticks(range(len(etiquetas))); ax.set_yticklabels(etiquetas)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.1f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, fraction=0.046); ax.set_title("Correlaciones de retornos mensuales (USD)")
plt.tight_layout(); plt.show()

In [ ]:
# Correlacion media dentro de cada grupo y entre grupos
def corr_media(a, b):
    bloque = corr.loc[a, b].values
    if a == b:
        return bloque[np.triu_indices(len(a), k=1)].mean()    # sin la diagonal
    return bloque.mean()

nombres = list(GRUPOS)
resumen = pd.DataFrame([[corr_media(GRUPOS[g1], GRUPOS[g2]) for g2 in nombres] for g1 in nombres],
                       index=nombres, columns=nombres)
resumen.round(2)

**Preguntas de discusión:** ¿hay alguna correlación negativa? ¿Qué pares se mueven más juntos y qué tienen en común (sector, país, materia prima)? Para un inversionista que ya tiene una minera peruana, ¿diversifica más otra minera, un banco local o una empresa de consumo de Estados Unidos? Mira la tabla por grupos: la correlación dentro de un mismo país o sector suele ser mayor que entre grupos. Esa es la razón de fondo para diversificar internacionalmente.

## 4. Dos activos con datos reales: BVN y KO

La curva de la clase, ahora con una minera de oro peruana y una empresa de consumo defensivo. Todos los insumos (medias, volatilidades y correlación) salen de la muestra.

In [ ]:
A, B = "BVN", "KO"
mu_a, mu_b = tabla.loc[A, "retorno medio"], tabla.loc[B, "retorno medio"]
s_a, s_b = tabla.loc[A, "volatilidad"], tabla.loc[B, "volatilidad"]
rho = corr.loc[A, B]
print(f"{A}: mu = {mu_a:.1%}, sigma = {s_a:.1%} | {B}: mu = {mu_b:.1%}, sigma = {s_b:.1%} | rho = {rho:.2f}")

pesos = np.linspace(0, 1, 101)
curva = np.array([portafolio_dos_activos(w, mu_a, mu_b, s_a, s_b, rho) for w in pesos])
recta = np.array([portafolio_dos_activos(w, mu_a, mu_b, s_a, s_b, 1.0) for w in pesos])

w_min = peso_minima_varianza(s_a, s_b, rho)
e_min, s_min = portafolio_dos_activos(w_min, mu_a, mu_b, s_a, s_b, rho)
print(f"Minima varianza: {w_min:.1%} en {A} y {1 - w_min:.1%} en {B} -> sigma = {s_min:.1%} (vs {min(s_a, s_b):.1%} del menos riesgoso solo)")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(curva[:, 1] * 100, curva[:, 0] * 100, color="#000080", lw=2.5, label=f"rho observada = {rho:.2f}")
ax.plot(recta[:, 1] * 100, recta[:, 0] * 100, color="gray", ls="--", label="rho = 1 (sin diversificacion)")
ax.scatter([s_a * 100, s_b * 100], [mu_a * 100, mu_b * 100], color="black", zorder=3)
ax.annotate(A, (s_a * 100, mu_a * 100), xytext=(6, 4), textcoords="offset points")
ax.annotate(B, (s_b * 100, mu_b * 100), xytext=(6, 4), textcoords="offset points")
ax.scatter(s_min * 100, e_min * 100, color="#006E6E", s=70, zorder=3, label="Minima varianza")
ax.set_xlabel("Volatilidad anualizada (%)"); ax.set_ylabel("Retorno medio anualizado (%)")
ax.set_title(f"Todas las mezclas de {A} y {B}"); ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

**Lectura:** la distancia horizontal entre la recta gris y la curva azul es riesgo que desapareció gratis. ¿El portafolio de mínima varianza tiene menos riesgo que el activo menos volátil por sí solo? Prueba otros pares cambiando `A` y `B`: ¿con cuál par la curva se dobla más y por qué? (Pista: mira la correlación.)

## 5. El experimento central: ¿aparece el piso?

La teoría dijo que con pesos iguales $\sigma_p^2 = \bar{\sigma}^2/n + \frac{n-1}{n}\,\overline{cov}$. Lo verificamos: para cada $n$ armamos 300 portafolios al azar de $n$ acciones del universo con pesos iguales, medimos su volatilidad y promediamos.

In [ ]:
cov = ret.cov() * 12                                       # covarianzas anualizadas
N = len(UNIVERSO)
rng = np.random.default_rng(421)

vol_media = []
for n in range(1, N + 1):
    vols = []
    for _ in range(300):
        elegidos = rng.choice(UNIVERSO, size=n, replace=False)
        w = np.full(n, 1 / n)
        vols.append(riesgo_portafolio(w, cov.loc[elegidos, elegidos]))
    vol_media.append(np.mean(vols))

# Los insumos de la formula teorica, medidos en nuestro universo
var_media = np.diag(cov).mean()
cov_media = cov.values[np.triu_indices(N, k=1)].mean()
sigma_media, rho_media = np.sqrt(var_media), cov_media / var_media
teorica = [riesgo_equiponderado(n, sigma_media, rho_media) for n in range(1, N + 1)]
piso = np.sqrt(cov_media)
print(f"Volatilidad media de una accion: {sigma_media:.1%} | correlacion media implicita: {rho_media:.2f} | piso: {piso:.1%}")

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(range(1, N + 1), np.array(vol_media) * 100, "o-", color="#000080", label="Simulacion (300 portafolios por n)")
ax.plot(range(1, N + 1), np.array(teorica) * 100, "--", color="#E36C0A", label="Formula de la clase")
ax.axhline(piso * 100, color="gray", ls=":", label=f"Piso sistematico ({piso:.1%})")
ax.set_xlabel("Numero de acciones en el portafolio"); ax.set_ylabel("Volatilidad anualizada (%)")
ax.set_title("El efecto diversificacion"); ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

**Lectura profesional:** ¿cuántos puntos de volatilidad se van al pasar de 1 a 5 acciones, y cuántos de 5 a 14? Las primeras acciones hacen casi todo el trabajo. La curva se aplana sobre el piso: esa es la parte del riesgo que todas comparten, y ningún número de acciones la elimina. (Detalle fino: en $n = 1$ la fórmula queda un poco por encima de la simulación porque promedia varianzas y luego saca la raíz, mientras la simulación promedia volatilidades; desde $n = 2$ coinciden.)

## 6. ¿Qué fracción del riesgo de cada acción es sistemática?

Un adelanto de la semana 9. Si regresas el retorno de una acción contra el del mercado, el $R^2$ (la correlación al cuadrado) es la fracción de su varianza que explica el mercado: la parte sistemática. El resto es idiosincrático y se puede diversificar.

In [ ]:
mercado = ret_todo["SPY"]
sistematico = pd.DataFrame({
    "beta vs SPY": ret.apply(lambda x: x.cov(mercado) / mercado.var()),
    "fraccion sistematica (R2)": ret.corrwith(mercado) ** 2,
})
sistematico["fraccion diversificable"] = 1 - sistematico["fraccion sistematica (R2)"]
sistematico.round(2).sort_values("fraccion sistematica (R2)", ascending=False)

**Para discutir:** en la mayoría de acciones individuales, más de la mitad de la varianza es diversificable. Por esa parte el mercado no paga prima: es riesgo que cargas gratis si no diversificas. ¿Por qué las acciones de la BVL salen con una fracción sistemática tan baja contra SPY? ¿Cambiaría si el "mercado" fuera EPU? (Pruébalo cambiando una línea.) La elección del índice de mercado importa, y volveremos a ella con el CAPM.

## 7. Cierre

Desde hoy quedan en la librería: `estadisticos_escenarios()`, `covarianza_escenarios()`, `portafolio_dos_activos()`, `peso_minima_varianza()`, `riesgo_portafolio()`, `riesgo_equiponderado()` y `anualizar()`.

**Tarea** (`clase06_tarea.ipynb`): tu empresa entra a un portafolio. Entrega hasta el lunes, vía commit.

**Próxima semana:** Markowitz. Hoy los pesos fueron iguales o los elegimos a mano; la próxima semana los elige un optimizador: frontera eficiente, mínima varianza global y portafolio tangente. Habrá sesión de reforzamiento rumbo al parcial de la semana 8.